## **Chemical Equilibrium: the $\rm{N_2O_4(g)} \rightleftharpoons 2 \; \rm{NO_2(g)}$ reaction**

<div align="left">
  <table border="1" cellpadding="6" cellspacing="0">
    <tr>
      <td bgcolor="#444444">
        <font color="#ffeb3b"><tt><b>Last updated (YYYY-MM-DD): 2026-04-16</b></tt></font>
      </td>
    </tr>
  </table>
</div>



In this notebook, we focus on a single, well-defined chemical system—the thermal dissociation of dinitrogen tetroxide into nitrogen dioxide—as a concrete example to explore the fundamental principles governing **chemical equilibrium**:

$$\rm N_2O_4 (g) \rightleftharpoons 2\,NO_2 (g)$$

This activity integrates the analysis of this equilibrium into a unified computational workflow. In particular, the equilibrium will be examined from three complementary perspectives: **classical thermodynamics**, which provides the macroscopic criteria for equilibrium through the minimization of thermodynamic potentials; **statistical thermodynamics**, which connects equilibrium properties with molecular-level information such as energies and partition functions; and **chemical kinetics**, which describes the time-dependent evolution of the reacting system and the dynamic approach to equilibrium.

The notebook is designed to guide the student through the quantitative description of equilibrium by examining how thermodynamic potentials vary with the extent of reaction, how equilibrium compositions emerge from minimization criteria under specified constraints, and how changes in temperature or external conditions influence the position of equilibrium. By working with a specific and chemically meaningful system, the notebook aims to provide an intuitive and computationally accessible introduction to equilibrium as a dynamic and quantitatively predictable state of matter, integrating thermodynamic, molecular, and kinetic viewpoints within a single computational framework.


### **Preparing the Computational Environment**  


*The next three cells prepare the environment by installing the necessary packages, importing libraries, and loading constants and functions. Make sure to run hem sequentially before doing anything else.*

⏳ **Note:** This may take **a few minutes** to complete. ⏳  

In [ ]:
#@title <small> 💻 Install Libraries (this may take a while) <small> { display-mode: "form" }
%%capture captured_output

# --- system (for LaTeX text in matplotlib) --- #
! sudo apt update -y
! sudo apt install -y cm-super dvipng texlive-latex-extra texlive-latex-recommended

# --- Python: install and update core scientific libraries --- #

# versions that are known to work well together will be selected to avoid compatibility issues
%pip install "numpy==2.0.2"
%pip install "scipy==1.16.3"

# install pyscf: as pyscf[geomopt,dispersion] gives problems, they are installed separately
%pip install --prefer-binary "pyscf==2.11.0"
%pip install "geometric==1.1"
%pip install "pyscf-dispersion==1.3.0"

%pip install "py3Dmol==2.5.3"
%pip install "pubchempy==1.0.5"
%pip install "rdkit==2025.9.1"

# !pip index versions pyscf      # check available versions
# %pip install "pyberny==0.6.3"  # install specific version
# !pip show pyberny              # check installed version

In [ ]:
#@title <small> 💻 Download subroutines.py with constants and subroutines from GitHub<small> { display-mode: "form" }

# Download the latest version of subroutines.py from the GitHub repository
# -q: run quietly (no verbose output)
# -O: save the file with the specified name (subroutines.py)
!wget -q -O subroutines.py https://raw.githubusercontent.com/daferro/Notebooks-Chemical_Equilibrium/main/subroutines.py

# Import the importlib module, which allows reloading already imported modules
import importlib

# Import the subroutines module (the downloaded Python file)
import subroutines as sr

# Reload the module to ensure that the most recent version is used
# This is important if the file was updated and the cell is executed again
importlib.reload(subroutines)

# Import all functions, constants, and variables from the module into the notebook namespace
from subroutines import *

# --------------------------------------------- #
def on_download_clicked(_,_case_):
    if _case_.startswith("kin"):
       init_conditions = f"{T_slider.value:.0f}K_{P_slider.value:.2f}bar_{V_slider.value:.2f}L_{yA_slider.value:.2f}"
    else:

    # update fig using last_fig
    # (a) from subroutine.py
       fig = sr.last_fig
    # (b) from this notebook
    for k in "kin,interc".split(","):
        if _case_.startswith(k): fig = last_fig

    # Can figure be downloaded?
    if fig is None:
       print("No figure yet; move a slider to generate the plot...")
       return

    # --- get file name ---
    if   _case_ == "PT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "VT"       : fname = f"equilibrium_T{T_slider.value:.0f}K_V{V_slider.value:.2f}L.svg"
    elif _case_ == "intercept": fname = f"intercept_T{T_slider.value:.0f}K_p{P_slider.value:.2f}bar.svg"
    elif _case_ == "dof"      : fname = "vib_dof.svg"
    elif _case_ == "kinPT"    : fname = f"chemkinPT_{init_conditions:s}.svg"
    elif _case_ == "kinVT"    : fname = f"chemkinVT_{init_conditions:s}.svg"
    else                      : raise Exception

    # if _case_.startswith("kin"): original_size = fig.get_size_inches()
    # if _case_.startswith("kin"): fig.set_size_inches(12,6)
    fig.savefig(fname, bbox_inches='tight',dpi=300)
    # if _case_.startswith("kin"): fig.set_size_inches(original_size)
    files.download(fname)
# --------------------------------------------- #


In [ ]:
#@title <small> 💻 Load Auxiliary Functions Specific for the N2O4/NO2 equilibrium <small> { display-mode: "form" }

NPOINTSYB = 51   # number of yB points (thermo & kinetics)
last_info = None
last_fig  = None

# ============================================= #
# ---- Specific for N2O4 <-> 2NO2 (PART 1) ---- #
# ============================================= #
def load_n2o4_2no2():
    # from Atkins' Physical Chemistry
    TREF     = 298
    DHo_ref  =  57.20 * 1E3
    DSo_ref  = 175.83
    DCPo_ref =  -2.88
    #-------------------------------------------
    refdata = (DHo_ref,DSo_ref,DCPo_ref,TREF)
    #-------------------------------------------
    molecules     = ["N2O4","NO2"]
    nus           = np.array([-1,2])
    #-------------------------------------------
    GEOMINFO      = {"N2O4":[(1,5),(4,5),(1,5,3)] , "NO2":[(0,1),(1,0,2)]}
    #-------------------------------------------
    n_0           = np.array([1.00, 0.00])
    #-------------------------------------------
    return refdata,molecules,nus,n_0,GEOMINFO
# --------------------------------------------- #
def get_xieq_PT_N2O4(T,P,nA0,nB0):
    refdata = load_n2o4_2no2()[0]
    dG0     = get_DGo(T,refdata)
    Kp      = np.exp(-dG0/R/T)
    alpha   = Kp * P_o/P
    # solution for second-order equation, knowing that (-nB0/2<= xi <= nA0)
    xi_eq   = (-nB0+(2*nA0+nB0)*np.sqrt(alpha/(alpha+4)))/2
    return xi_eq
# --------------------------------------------- #
def get_xieq_VT_N2O4(T,V,nA0,nB0):
    refdata = load_n2o4_2no2()[0]
    dG0     = get_DGo(T,refdata)
    Kp      = np.exp(-dG0/R/T)
    beta    = Kp * (P_o*V/R/T)
    # solution for second-order equation, knowing that (-nB0/2<= xi <= nA0)
    xi_eq   = (-4*nB0-beta+np.sqrt((8*(2*nA0+nB0)+beta)*beta))/8
    return xi_eq
# --------------------------------------------- #
def yB_to_xi_N2O4(yB):
    refdata,molecules,nus,n_0,GEOMINFO = load_n2o4_2no2()
    xi = (yB*n_0.sum() - n_0[1])/(2-yB)
    return xi
# --------------------------------------------- #
def intercept_getGm_N2O4(T,P,yB):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]
    xi     = yB_to_xi_N2O4(yB)
    ntot   = n_0.sum() + xi
    Gtot   = get_G_PT(xi,P,T,n_0,nus,refdata)
    Gm     = Gtot/ntot
    return Gm, Gtot, ntot
# --------------------------------------------- #
def intercept_getline_N2O4(T,P,yB_i):
    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]
    #partial pressures and quotient of reaction
    yA_i  = 1-yB_i
    pB_i  = P*yB_i
    pA_i  = P*yA_i
    Qp    = (pB_i/P_o)**2 / (pA_i/P_o)
    # value for xi_i
    xi_i  = yB_to_xi_N2O4(yB_i)
    # delta_r G^o (T)
    dGo_T = get_DGo(T,refdata)
    # G_tot
    Gm_i, Gtot_i, ntot_i = intercept_getGm_N2O4(T,P,yB_i)
    # get slope (m)
    dxidyB = (2*n_0.sum() - n_0[1]) / (2-yB_i)**2
    dGdxi  = dGo_T + R*T*np.log(Qp)
    m      = 1/ntot_i * dxidyB * (dGdxi - Gm_i)
    # Point of the line
    xx     = yB_i
    yy     = (Gm_i - intercept_getGm_N2O4(T,P,0)[0])/(R*T)
    # get intercept (b)
    m      = m / (R*T)
    b      = yy - m*xx
    return m,b, (xx,yy)
# --------------------------------------------- #
def plot_intercept_N2O4(T,P,yB):
    nA0,nB0 = load_n2o4_2no2()[3]
    # equilibrium
    xi_eq   = get_xieq_PT_N2O4(T,P,nA0,nB0)
    yB_eq   = (n_0[1]+2*xi_eq)/(n_0.sum() + xi_eq)
    Gtot_eq = intercept_getGm_N2O4(T,P,yB_eq)[1]
    # Get data in terms of yB_i
    lGm,lGtot,lntot = [],[],[]
    list_yB         = np.linspace(0.0,1.0,NPOINTSYB)
    for yB_i in list_yB:
        Gm_i, Gtot_i, ntot_i = intercept_getGm_N2O4(T,P,yB_i)
        lntot.append(ntot_i)
        lGtot.append(Gtot_i)
    # control point
    if yB == 0.0: yB = ZERO4
    if yB == 1.0: yB = 1 - ZERO4
    # equation for tangent of Gm
    m,b,(xx1,yy1) = intercept_getline_N2O4(T,P,yB)
    # values at the selected point
    Gm, Gtot, ntot = intercept_getGm_N2O4(T,P,yB)
    # apply reference
    Gtot    =   (Gtot    - lGtot[0])/(R*T)
    Gtot_eq =   (Gtot_eq - lGtot[0])/(R*T)
    lGtot   = [(lGtot[i] - lGtot[0])/(R*T) for i in range(len(lGtot))]
    lGm     = [lGtot[i]/lntot[i]           for i in range(len(lGtot))]
    # --- Create side-by-side axes ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    # -- left plot --
    ax1.plot(list_yB,lGtot,'k-')
    ax1.plot(yB_eq,Gtot_eq,'ko',zorder=1)
    ax1.plot(yB   ,Gtot   ,'ro',zorder=2)
    for ii in ["x","y"]: ax1.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax1.set_xlim( 0.0 , 1.0 )
    ax1.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax1.set_ylabel(r"$[G(y_{\rm NO_2})-G(0)]\cdot (RT)^{-1} \;\; \mathrm{[mol]}$",fontsize=FONTSIZE[2])
    # -- right plot --
    xx2 = [ZERO4,1-ZERO4]
    yy2 = [m*xx_i + b for xx_i in xx2]
    ylim1 = min(lGm)
    ylim2 = max(lGm)
    delta = ylim2-ylim1
    ylim1 = ylim1 - delta*0.1
    ylim2 = ylim2 + delta*0.1
    ax2.plot(list_yB,lGm,'k-')
    ax2.plot(xx1,yy1,'ro')
    ax2.plot(xx2,yy2,'r--',label=rf"${m:+.5f} \cdot y_{{\rm NO_2}} {b:+.5f}$")
    for ii in ["x","y"]: ax2.tick_params(axis=ii,labelsize=FONTSIZE[1])
    ax2.set_xlim( 0 , 1 )
    ax2.set_ylim(ylim1,ylim2)
    ax2.set_xlabel(r"$y_{\rm NO_2}$",fontsize=FONTSIZE[2])
    ax2.set_ylabel(r"$[G_{\rm m}(y_{\rm NO_2})- G_{\rm m}(0)] \cdot (RT)^{-1}$",fontsize=FONTSIZE[2])
    ax2.legend(loc="upper center",fontsize=FONTSIZE[0])
    # --- update global variable: last_fig ---
    global last_fig
    last_fig = plt.gcf()
    # --- Show and close figure ---
    plt.show()
    plt.close()
    print(rf"Equilibrium at y(NO2) = {yB_eq:.7f}")
# --------------------------------------------- #
def plot_3DeqPT_N2O4(T,P):

    refdata,molecules,nus,n_0 = load_n2o4_2no2()[0:4]

    # calculate data for each plot
    Z1 = get_xieq_PT_N2O4(T,P,n_0[0],n_0[1])
    Z2 = get_G_PT(Z1,P,T,n_0,nus,refdata)/(R*T)

    # Create figure with two subplots
    # title1 = r'$(a) \;\; z: \;\; \xi_{\mathrm{eq}} \;\; \text{(mol)}$'
    # title2 = r'$(b) \;\; z: \;\; [G(\xi_{\mathrm{eq}}) - G(0)] / RT  \;\; \text{(mol)}$'
    title1 = "(a)  z:  <i>ξ</i><sub>eq</sub> (mol)"
    title2 = "(b)  z:  [<i>G</i>(<i>ξ</i><sub>eq</sub>) − <i>G</i>(0)] / <i>RT</i>  (mol)"
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'surface'}, {'type': 'surface'}]],
        subplot_titles=(title1,title2))

    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z1, colorscale='Viridis', showscale=False),row=1, col=1)
    fig.add_trace(go.Surface(x=P/1E5, y=T, z=Z2, colorscale='Plasma' , showscale=False),row=1, col=2)

    # formating
    fig.update_layout(
        width=1000, height=450,
        margin=dict(l=0, r=0, t=0, b=0),
        scene=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.00, 0.50], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9))),
        scene2=dict(
          xaxis=dict(title=dict(text='p (bar)',font=dict(size=18))),
          yaxis=dict(title=dict(text='T (K)'  ,font=dict(size=18))),
          zaxis=dict(title=dict(text=''       ,font=dict(size=18))),
          domain=dict(x=[0.50, 1.00], y=[0, 1]),
          camera=dict(eye=dict(x=1.6, y=1.6, z=0.9)))
    )

    # Ajust position of plot titles
    for ann in fig['layout']['annotations']:
        ann['y'] -= 0.15
        ann['font'] = dict(size=16)

    fig.show(config={"toImageButtonOptions": {"format": "svg","filename": "equilibrium_surfaces","scale": 2}})
# ============================================= #

# ============================================= #
# ---- Specific for N2O4 <-> 2NO2 (PART 3) ---- #
# ============================================= #
def get_constants_N2O4(T):
    # Gibbs free energy from experimental data
    DGo   = get_DGo(T,load_n2o4_2no2()[0])
    # Equilibrium constant(s)
    Kp_o  = np.exp(-DGo/R/T)
    Kc_o  = Kp_o * P_o/(c_o*R*T) # adimensional
    Kc    = Kc_o * c_o           # mol/m^3
    # Forward rate constant
    kfw   = arrhenius_A * np.exp(-arrhenius_B/T)
    # Backward rate constant
    kbw  = kfw/Kc
    # String with data
    string  = rf"   * Constants for the reaction at {T:.2f} K" + "\n"
    string += "\n"
    string += rf"     equilibrium constant   Kp^o = {Kp_o:.3E}" + "\n"
    string += rf"     equilibrium constant   Kc^o = {Kc_o:.3E}" + "\n"
    string += rf"     forward  rate constant kfw  = {kfw:.3E} 1/s" + "\n"
    string += rf"     backward rate constant kbw  = {kbw:.3E} m^3 / mol / s" + "\n"
    string += "\n"
    # return data
    return DGo,Kp_o,Kc_o,kfw,kbw,string
# --------------------------------------------- #
def xi_to_data_N2O4(xi,T0,P0,V0,yA0,scenario):
    # values at xi=0
    n0 = P0*V0/(R*T0)
    nA0 = yA0*n0
    nB0 = n0-nA0
    # values at xi
    nA = nA0 -   xi
    nB = nB0 + 2*xi
    n  = n0  +   xi
    yA = nA/n
    yB = nB/n
    if   "VT" in scenario: P,V = n*R*T0/V0, V0
    elif "PT" in scenario: P,V = P0 , n*R*T0/P0
    else: raise Exception
    cA = nA/V
    cB = nB/V
    pA = yA*P
    pB = yB*P
    # quotient of reaction
    Qp = np.where(pA == 0,np.inf,(pB/P_o)**2 / (pA/P_o))
    # energy of interest
    if "VT" in scenario: energy = get_A_VT_N2O4(xi,V0,T0,nA0,nB0)
    if "PT" in scenario: energy = get_G_PT_N2O4(xi,P0,T0,nA0,nB0)
    # refdata = load_n2o4_2no2()[0]
    # if "PT" in scenario: energy = get_G_PT(xi,P0,T0,np.array([nA0,nB0]),np.array([-1,2]),refdata)
    # if "VT" in scenario: energy = get_A_VT(xi,V0,T0,np.array([nA0,nB0]),np.array([-1,2]),refdata)
    # return data
    return (nA,nB),(pA,pB),(yA,yB),(cA,cB),(n,P,V),Qp,energy
# --------------------------------------------- #
def xi2time_PT_N2O4(xi,xi1,xi2,kfw,alpha):
    beta  = kfw + 4*alpha
    s     = np.sqrt(kfw/beta)
    term1 = (xi1-xi)/xi1
    term2 = (xi2-xi)/xi2
    texp  = term1**(1+s) / term2**(1-s)
    t     = -np.log(texp)/(2*s*beta)
    return t
# --------------------------------------------- #
def xi2time_VT_N2O4(xi,xi1,xi2,kbw,V0):
    t  = np.log( abs(xi1/xi2 * (xi-xi2)/(xi-xi1))  )
    t *= V0/(4*kbw*(xi1-xi2))
    return t
# --------------------------------------------- #
def get_G_PT_N2O4(xi,P,T,nA0,nB0):
    #Delta_r{G}^o(T)
    DGo_T   = get_constants_N2O4(T)[0]
    #Delta_r{G}^*(T)
    DGast   = DGo_T + R*T*np.log(P/P_o)
    # mix term
    n0      = nA0 + nB0
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    DDGmix  = 0.0
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return G(xi) - G(0)
    DGtot   = DGast * xi + DDGmix
    return DGtot
# --------------------------------------------- #
def get_A_VT_N2O4(xi,V,T,nA0,nB0):
    # initial conditions
    n0      = nA0 + nB0
    p0      = n0*R*T/V
    # current conditions
    nA      = nA0 -   xi
    nB      = nB0 + 2*xi
    n       = nA  + nB
    p       = n *R*T/V
    #Delta_r{G}^o(T)
    DGo_T   = get_constants_N2O4(T)[0]
    # pressure term
    termP   = p *np.log(p/P_o/np.e)
    termP  -= p0*np.log(p0/P_o/np.e)
    # mix term
    DDGmix  = 0.0
    if nA  != 0.0: DDGmix += nA *np.log(nA /n )
    if nB  != 0.0: DDGmix += nB *np.log(nB /n )
    if nA0 != 0.0: DDGmix -= nA0*np.log(nA0/n0)
    if nB0 != 0.0: DDGmix -= nB0*np.log(nB0/n0)
    DDGmix *= R*T
    # return A(xi) - A(0)
    DAtot   = DGo_T * xi + V*termP + DDGmix
    return DAtot
# --------------------------------------------- #
def datatoinfo_N2O4(T0,p0,V0,yA0,xi,scenario):
    # constants
    Kp_o = get_constants_N2O4(T0)[1]
    # calculate more magnitudes
    (nA,nB),(pA,pB),(yA,yB),(cA,cB),(n,P,V),Qp,E = xi_to_data_N2O4(xi,T0,p0,V0,yA0,scenario)
    # string with information
    string  = rf"       (P , V , T) = ({P*1E-5:6.2f} bar , {V*1E3:6.2f} L , {T0:6.2f} K)"+"\n"
    string += "\n"
    string += rf"       num. moles  = {n:6.3f} mol"+"\n"
    string += rf"       extent (xi) = {xi:6.3f} mol"+"\n"
    string += "\n"
    if "VT" in scenario: string += rf"       A(xi)-A(0)  = {E/(R*T0):8.2E}*(RT) mol"+"\n"
    if "PT" in scenario: string += rf"       G(xi)-G(0)  = {E/(R*T0):8.2E}*(RT) mol"+"\n"
    string += "\n"
    string += rf"       data for N2O4"+"\n"
    string += rf"         - number of moles  = {nA:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yA:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pA*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cA/1000:8.2E} M"+"\n"
    string += "\n"
    string += rf"       data for NO2"+"\n"
    string += rf"         - number of moles  = {nB:6.3f} mol"+"\n"
    string += rf"         - mole fraction    = {yB:6.3f} mol"+"\n"
    string += rf"         - partial pressure = {pB*1E-5:6.3f} bar"+"\n"
    string += rf"         - concentration    = {cB/1000:8.2E} M"+"\n"
    string += "\n"
    if pA == 0.0: string += rf"       ==> Qp^o = infinity"+"\n"
    else        : string += rf"       ==> Qp^o = {(pB*pB)/(pA*P_o):.3E}"+"\n"
    string += rf"       ==> Kp^o = {Kp_o:.3E}"+"\n"
    string += "\n"
    return string
# --------------------------------------------- #
def kinetics_N2O4(T0,P0,V0,yA0,scenario):
    # --- global variable to update ---
    global last_info
    # which scenario
    if   "PT" in scenario: scenario = "PT"
    elif "VT" in scenario: scenario = "VT"
    else: raise Exception
    # Get data for this reaction
    DGo,Kp_o,Kc_o,kfw,kbw,STRING = get_constants_N2O4(T0)
    # Determine xieq, xi1 and xi2
    (nA0,nB0) = xi_to_data_N2O4(0,T0,P0,V0,yA0,scenario)[0]
    if scenario == "VT":
       xieq = get_xieq_VT_N2O4(T0,V0,nA0,nB0)
       Kc   = kfw/kbw
       lamb = 4*(kbw/V0)
       a0   = (nB0**2/4 - nA0*V0*Kc/4)
       a1   = (nB0      +     V0*Kc/4)
       xi1  = (-a1 + np.sqrt(a1**2-4*a0))/2
       xi2  = (-a1 - np.sqrt(a1**2-4*a0))/2
    if scenario == "PT":
       xieq  = get_xieq_PT_N2O4(T0,P0,nA0,nB0)
       alpha = kbw*P0/(R*T0)
       beta  = kfw + 4*alpha
       s     = np.sqrt(kfw/beta)
       xi1   = (-nB0+(2*nA0+nB0)*s)/2
       xi2   = (-nB0-(2*nA0+nB0)*s)/2
    # Calculate from xi = 0 to xieq
    xis    = np.linspace(0,REL_XI_EQ*xieq,NPOINTSXI)
    if scenario == "VT": times = [xi2time_VT_N2O4(xi,xi1,xi2,kbw,V0   ) for xi in xis]
    if scenario == "PT": times = [xi2time_PT_N2O4(xi,xi1,xi2,kfw,alpha) for xi in xis]
    # for t_i,xi_i in zip(times,xis): print(t_i,xi_i)
    # String with information
    STRING += rf"   * Initial conditions:"+"\n\n"
    STRING += datatoinfo_N2O4(T0,P0,V0,yA0,0   ,scenario)
    STRING += rf"   * At equilibrium:"+"\n\n"
    STRING += datatoinfo_N2O4(T0,P0,V0,yA0,xieq,scenario)
    last_info = STRING
    # plot data
    plot_kinetics_N2O4(np.array(times),xis,xieq,T0,P0,V0,yA0,scenario)
# ============================================= #
def plot_kinetics_N2O4(times,xis,xieq,T0,P0,V0,yA0,scenario):

    # select good units for time (among secs, milisecs, microsecs and nanosecs)
    for unitst,factor in [("s",1E0) , ("ms",1E3) , ("$\\mu$s",1E6) , ("ns",1E9)]:
        last_t = times[-1]*factor
        if last_t > 0.5: break

    dataeq = xi_to_data_N2O4(xieq,T0,P0,V0,yA0,scenario)
    yA,yB = [],[]
    Qp    = []
    AG    = []
    V     = []
    P     = []
    nA    = []
    for xi_i in xis:
        tni,tpi,tyi,tci,(n_i,P_i,V_i),Qp_i,E_i = xi_to_data_N2O4(xi_i,T0,P0,V0,yA0,scenario)
        nA.append(tni[0])
        yA.append(tyi[0])
        yB.append(tyi[1])
        Qp.append(Qp_i)
        AG.append(E_i)
        V.append(V_i)
        P.append(P_i)

    plt.rcParams['text.usetex'] = True
    fig, axs = plt.subplots(2, 3, figsize=(12,6))
    fig.suptitle(rf'$(P,V,T)_0 = ({P0*1E-5:.2f} \; {{\rm bar}},{V0*1E3:.2f} \; {{\rm L}},{T0:.2f} \; {{\rm K}})$; $y_{{\rm N_2O_4}}(0)={yA0:.2f}$', fontsize=FONTSIZE[2])
    # -------------------------------------
    # (a) Population
    # -------------------------------------
    axs[0, 0].plot(times*factor,yA,color='k',label=r'i=N$_2$O$_4$')
    axs[0, 0].axhline(y=dataeq[2][0],color="k",ls=":",zorder=1)

    axs[0, 0].plot(times*factor,yB,color='r',label=r'i=NO$_2$')
    axs[0, 0].axhline(y=dataeq[2][1],color="r",ls=":",zorder=1)

    axs[0, 0].yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    axs[0, 0].set_ylabel(r'$y_i$',fontsize=FONTSIZE[2])
    axs[0, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 0].set_title('(a)')
    axs[0, 0].legend(frameon=False)

    # -------------------------------------
    # (b) ξ vs time
    # -------------------------------------
    nA0,nB0 = xi_to_data_N2O4(0,T0,P0,V0,yA0,scenario)[0]
    xlim1   = -nB0/2
    xlim2   = +nA0
    axs[0, 1].plot(times*factor,xis,ls='-',color='k')
    axs[0, 1].axhline(y=xieq,ls=":",color="k",zorder=1)

    axs[0, 1].set_ylabel('$\\xi$ (mol)',fontsize=FONTSIZE[2])
    # axs[0, 1].set_ylim(xlim1,xlim2)
    axs[0, 1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

    axs[0, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 1].set_title('(b)')

    # -------------------------------------
    # (c) P(or V) vs time
    # -------------------------------------
    if "PT" in scenario:
        axs[0, 2].plot(times*factor,np.array(V)*1E3 ,ls='-',color='k')
        axs[0, 2].set_ylabel('$V$ (L)',fontsize=14)
    if "VT" in scenario:
        axs[0, 2].plot(times*factor,np.array(P)*1E-5,ls='-',color='k')
        axs[0, 2].set_ylabel('$P$ (bar)',fontsize=FONTSIZE[2])
    axs[0, 2].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[0, 2].set_title('(c)')

    # -------------------------------------
    # (d) Q vs time
    # -------------------------------------
    xx,yy=factor*times[1:], Qp[1:]
    axs[1, 0].plot(xx,yy,ls='-',color='k',zorder=2)
    axs[1, 0].axhline(y=dataeq[5],ls=":" ,color="k",zorder=1)
    axs[1, 0].set_ylabel('$Q_p^\\circ$',fontsize=FONTSIZE[2])
    axs[1, 0].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 0].set_title('(d)')

    # -------------------------------------
    # (e) dξ/dt vs time
    # -------------------------------------
    Kpo,Kco,kfw = get_constants_N2O4(T0)[1:4]
    dxidt_ana       = kfw*np.array(nA)*(1-Qp/Kpo)
    axs[1, 1].plot(times*factor,dxidt_ana/factor,ls='-',color='k')
    # dxidt_num       = np.gradient(xis,times)
    # axs[1, 1].plot(times*factor,dxidt_num/factor,'rx')
    axs[1, 1].set_ylabel(rf'd$\xi$/d$t$ (mol/{unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 1].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 1].set_title('(e)')

    # -------------------------------------
    # (f) G or A vs time
    # -------------------------------------
    yy = [Ei/(R*T0) for Ei in AG]
    axs[1, 2].plot(times*factor,yy,color='k')
    axs[1, 2].axhline(y=dataeq[6]/(R*T0),ls=":",color="k",zorder=1)
    if "PT" in scenario: key = "G"
    if "VT" in scenario: key = "A"
    axs[1, 2].set_ylabel(rf'$\left({key:s}(\xi)-{key:s}(0)\right) / (RT)$ (mol)',fontsize=FONTSIZE[2])
    axs[1, 2].set_xlabel(rf'Time ({unitst:s})',fontsize=FONTSIZE[2])
    axs[1, 2].set_title('(f)')

    # --- update global variable: last_fig ---
    fig.subplots_adjust(wspace=0.3)
    plt.tight_layout()
    global last_fig
    last_fig = fig
    # --- Show and close figure ---
    # fig.set_size_inches(9.0,6.6)
    plt.show()
    plt.close(fig)
# ============================================= #

### **Loading the reaction**  


Execute the following cell to load the reaction.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
MOLECULES,NUS,n_0 = load_n2o4_2no2()[1:4]
#------------------------------------------------------
string = reaction_to_string(NUS,MOLECULES)
#------------------------------------------------------
print(rf"The reaction to study is: {string:s}")
print("")
print(rf"Initial number of moles considered for each species:")
for molecule,n0_i in zip(MOLECULES,n_0):
    print(rf"   * {molecule:4s}: {n0_i:.3f} mol")
print("")
#------------------------------------------------------
TMIN = 200
TREF = load_n2o4_2no2()[0][-1]
TMAX = 400
assert TMIN <= TREF <= TMAX
#------------------------------------------------------

### **1. A Chemical Thermodynamics Perspective**  


####
At 298 K, the following thermodynamic data are available for the reaction (see Atkins' *Physical Chemistry*, **2014**, Oxford University Press):
-  for $\rm N_2O_4$ $\Rightarrow$ $\Delta_f{H}^\circ = \;\; 9.16$ kJ mol$^{-1}$, $S^\circ = 304.29$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 77.28$ J mol$^{-1}$ K$^{-1}$;
-  for $\rm NO_2$ $\;$ $\Rightarrow$ $\Delta_f{H}^\circ = 33.18$ kJ mol$^{-1}$, $S^\circ = 240.06$ J mol$^{-1}$ K$^{-1}$ and $C_{p,m}^\circ = 37.20$ J mol$^{-1}$ K$^{-1}$.

From these values, we obtain:
- $\Delta_{r} H^{\circ}(T_{\rm ref}) = \;\;57.20$ kJ mol$^{-1}$,
- $\Delta_{r} S^{\circ}(T_{\rm ref}) \;= 175.83$ J mol$^{-1}$ K$^{-1}$ and
- $\Delta_{r} C_p^\circ(T_{\rm ref}) \;= -2.88$ J mol$^{-1}$ K$^{-1}$.

Load this data by executing the cell below.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
REFDATA = load_n2o4_2no2()[0]
#------------------------------------------------------
print(rf"Magnitudes of interest at Tref = {TREF:.2f} K:")
print("")
print(rf"    Delta_r{{H}}^o  = {REFDATA[0]/1000:+8.2f} kJ/mol")
print(rf"    Delta_r{{S}}^o  = {REFDATA[1]:+8.2f}  J/(mol K)")
print(rf"    Delta_r{{Cp}}^o = {REFDATA[2]:+8.2f}  J/(mol K)")
print("")
#------------------------------------------------------

####
**(a) EFFECT OF THE TEMPERATURE ON $\Delta_r G^\circ$**

#####
For an ideal gas-phase reaction, the temperature dependence of the standard Gibbs free energy change, $\Delta_{r} G^\circ$, can be expressed as:

$$
\Delta_{r} G^{\circ}(T)= \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
 \tag{1}
$$

with $T_{\rm ref}$ being the reference $T$ and under the assumption that $\Delta_{r} C_p^\circ$ is independent of $T$. Notice that the corresponding equilibrium constant is then obtained as:

$$
K_p^\circ(T) = e^{-\Delta_{r} G^{\circ}(T)/(R \cdot T)}
 \tag{2}
$$

Examine the temperature dependence of $\Delta G^\circ$ (and of $K_p$) by executing the next cell. In the generated plots, the red dot indicates the value at $T_{\rm ref}$.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
T     = np.linspace(TMIN,TMAX,num=NPOINTST)
DGo_T = plot_DGo_T(T,TREF,REFDATA)
#------------------------------------------------------

#####
According to the Gibbs–Helmholtz equation, the following equality should hold:

$$
\frac{\Delta_{r} H^{\circ}}{T^2} = \frac{d}{dT}\left( -\frac{\Delta_{r} G^{\circ}}{T} \right)
\tag{3}
$$

This relationship can be tested by computing the numerical derivative of the term on the right-hand side and comparing it with the analytical expression on the left-hand side.

- The analytical expression for the enthalpy term as a function of temperature is:
$$
\Delta_{r} H^{\circ} = \Delta_{r} H^{\circ}(T_{\rm ref}) + \Delta_{r} C_{p}^\circ \cdot (T- T_{\rm ref})
\tag{4}
$$
Using this expression, we can easily plot $\Delta_{r} H^{\circ}/ T^2$.

- On the other side, the numerical derivative of $-\Delta_{r} G^{\circ}/T$ with respect to temperature can be directly computed from the data obtained in the execution of the previous cell.

Execute the next cell to check the Gibbs–Helmholtz relationship.

In [ ]:
#@title <small><small> { display-mode: "form" }

plot_gibbshelmholtz(T,DGo_T,REFDATA)

####
**(b) REACTION CONDUCTED AT CONSTANT _P_ AND _T_**

#####
$G$ changes dynamically as the reaction progresses. By knowing the initial conditions for the reaction, it is possible to track how $G$ varies with the extent of reaction ($\xi$), which is of special interest when the reaction takes place at constant $T$ and $p$. We will assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$).

Execute the cell below, set the temperature and the total pressure, and analyze how $G$ changes with $\xi$:

In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.linspace(xi_min,xi_max,num=NPOINTSXI)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=TREF,min=TMIN,max=TMAX,step=1.00,description=r'T [K]'  , **args)
P_slider = w.FloatSlider(value=1.00,min=0.01,max=3.00,step=0.01,description=r'p [bar]', **args)
ui       = w.VBox([T_slider,P_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"PT"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, fixed_args: plot_DG_PT(T, P*1E5, fixed_args), {'T': T_slider, 'P': P_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

#####
As you have observed, the equilibrium extent (and its corresponding Gibbs free energy) depends on the experimental temperature and pressure. This dependence can be visualized using a 3D plot, which enables us to examine (a) the equilibrium position, $\xi_{\rm eq}$, and (b) the change in Gibbs free energy from the initial state ($\xi = 0$) to the equilibrium state ($\xi_{\rm eq}$).

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------
Ts    = np.linspace(TMIN  ,TMAX   ,101)
Ps    = np.linspace(0.01E5, 3.01E5,101)
Ps,Ts = np.meshgrid(Ps, Ts)
#------------------------------------------------------
plot_3DeqPT_N2O4(Ts,Ps)
#------------------------------------------------------

#####

Beyond the previous global view, it is also useful to analyze the equilibrium condition directly in terms of mixture composition through the molar Gibbs free energy, defined as

$$
G_m(\xi) =  G(\xi) / n(\xi) \tag{5}
$$

where both the total Gibbs free energy, $G$, and the total number of moles, $n$, depend on the extent of reaction.

For the binary mixture considered here, this can also be written as:

$$
G_m(\xi) =  y_{N_2O_4} \cdot \mu_{N_2O_4}  + y_{NO_2} \cdot \mu_{NO_2} \tag{6}
$$

By plotting $G_m$ as a function of $y_{NO_2}$, the equilibrium composition can also be identified. Notably, the equilibrium point does not coincide with the minimum of this curve. Execute the following cell and use the $y_{NO_2}$ slider to explore how the equilibrium point can be inferred from the $G_m$ plot (hint: it is related to the _intercept method_).

In [ ]:
#@title <small><small> { display-mode: "form" }

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True)
T_slider = w.FloatSlider(value=TREF,min=TMIN,max=TMAX,step=1.0000  ,description=r'T [K]'  ,readout_format='.2f',**args)
P_slider = w.FloatSlider(value=1.00,min=0.01,max=3.00,step=0.0100  ,description=r'p [bar]',readout_format='.2f',**args)
yB_slider= w.FloatSlider(value=0.50,min=0.00,max=1.00,step=0.000001,description=r'y(NO2)' ,readout_format='.6f',**args)

# --- Separator text between slider groups ---
separator = w.HTML(
    value=(
        "<div style='width:600px; margin:10px 0 6px 0;'>"
        "  <hr style='border:0; border-top:1px solid rgba(255,255,255,0.25); margin:0 0 8px 0;'>"
        "  <div style='text-align:center; font-size:13px; line-height:1.25; opacity:0.85;'>"
        "    Move <b>y(NO2)</b> to select a point on <b>G<sub>m</sub></b>.<br>"
        "    The <b>tangent</b> to <b>G<sub>m</sub></b> will be shown."
        "  </div>"
        "</div>"
    )
)

# sliders
ui = w.VBox([T_slider, P_slider,separator,yB_slider],
            layout=w.Layout(width='600px',display='flex',flex_flow='column',align_items='flex-start',justify_content='flex-start'))

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"intercept"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda T, P, yB: plot_intercept_N2O4(T,P*1E5,yB), {'T': T_slider, 'P': P_slider, 'yB': yB_slider})
display(w.VBox([ui, btn]), out)

####
**(c) REACTION CONDUCTED AT CONSTANT _V_ AND _T_**

#####
Another case of interest is a reaction carried out in a reactor at constant $T$ and $V$. Under these conditions, the relevant thermodynamic potential is the Helmholtz free energy (A):

$$
A = G - pV = G - nRT
\tag{7}
$$

As in the previous case, let us assume that the reaction vessel initially contains 1 mol of N$_2$O$_4$ (and none of NO$_2$). Execute the cell below, set the temperature and the total volume, and analyze how $A$ changes with $\xi$:


In [ ]:
#@title <small><small> { display-mode: "form" }

#----  Get limit values for xi  ----
xi_min,xi_max = limits_xi(n_0,NUS)
#----     Get values for xi     ----
xis           = np.linspace(xi_min,xi_max,num=NPOINTSXI)
#----   Save info in a tuple    ----
fixed_args    = (MOLECULES,NUS,n_0,xis,REFDATA)
#-----------------------------------

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args     = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider = w.FloatSlider(value=TREF ,min=TMIN,max=TMAX  ,step=1.00,description=r'T [K]', **args)
V_slider = w.FloatSlider(value=24.78,min=2.00,max=250.00,step=0.01,description=r'V [L]', **args)
ui       = w.VBox([T_slider,V_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"VT"))

# -------- slider ---> function --------
# V is converted from L to m3 with *1E-3
out = w.interactive_output(lambda T, V, fixed_args: plot_DA_VT(T, V*1E-3, fixed_args), {'T': T_slider, 'V': V_slider, 'fixed_args': w.fixed(fixed_args)})
display(w.VBox([ui, btn]), out)

### **2. A Statistical-Mechanical Perspective**  


#####
In the previous section, we analyzed the equilibrium using tabulated values of $\Delta_{r} H^{\circ}(T_{\rm ref})$, $\Delta_{r} S^{\circ}(T_{\rm ref})$ and $\Delta_{r} C_p^\circ(T_{\rm ref})$, which were then inserted into a macroscopic expression for $\Delta_{r} G^{\circ}(T)$. In this section, we will obtain these thermodynamic quantities from a molecular-level description using statistical mechanics.

For an ideal-gas mixture, the standard thermodynamic functions of each species can be derived from its molecular partition function, $q^\circ(T)$, which factorizes into translational, rotational, vibrational, and electronic contributions. From $q^\circ(T)$ we can compute $G^{\circ}_{m}(T)$, $H^{\circ}_{m}(T)$, $S^{\circ}_{m}(T)$, and $C^{\circ}_{p,m}(T)$ for each molecule. The reaction quantities then follow by applying the stoichiometric sum over reactants and products. In this way, the temperature dependence of
$\Delta_{r} G^{\circ}(T)$ is linked directly to molecular structure and vibrational spectra.

To start, we need initial molecular geometries for the reactant (N$_2$O$_4$) and product (NO$_2$). These structures will be used as input for subsequent electronic-structure calculations (geometry optimization and harmonic vibrational analysis), from which we will extract vibrational frequencies and other molecular parameters required to build the partition functions.

#### **(a) Guess geometries of reactants and products**

We will retrieve the geometry for N$_2$O$_4$ from PubChem[[🌐]](https://pubchem.ncbi.nlm.nih.gov/) by using its PubChem CID identifier[[🌐]](https://pubchem.ncbi.nlm.nih.gov/compound/25352):

* PubChem CID for N$_2$O$_4$ is "25352"

Run the cell below and enter this identifier. Do _not_ include quotation marks when entering the CID.
Once the geometry is retrieved, it will be displayed in an interactive viewer that allows you to rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

molecule = "N2O4"
GEOMINFO = load_n2o4_2no2()[4]
# ------------------------------------------------
if "MOLECULES_ID" not in globals(): MOLECULES_ID  = {k:None for k in MOLECULES}
# ------------------------------------------------
print("Fetching structures from PubChem 🔎 .\n")
print("Please enter the CID to search for:")
MOLECULES_ID[molecule] = input(rf"     * reactant ({molecule:s}): ").strip()
print("")

mol_id    = MOLECULES_ID[molecule]
xyz_guess = files_of_interest(molecule)[0]

# Get geometry
print(rf"Retrieving geometry for: {str(mol_id):s}")
print("")
symbols,coords,smiles = pubchem_cid(mol_id)

# Generate file
if symbols is None:
   print(rf"     - ERROR: unable to get geometry for: '{str(mol_id):s}'")
   print("")
else:
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

#####

For NO$_2$ we will use an alternative way to obtain a reference structure: building the molecule from its SMILES representation[[🌐]](https://en.wikipedia.org/wiki/Simplified_Molecular_Input_Line_Entry_System) using with the **Rdkit** toolkit[[🌐]](https://www.rdkit.org/). You can find the SMILES for nitrogen dioxide online (including on Wikipedia[[🌐]](https://en.wikipedia.org/wiki/Nitrogen_dioxide)):

*  SMILES for NO$_2$: "[N+]\(=O)[O-]":

Run the following cell and paste the SMILES string to reconstruct a **3D geometry** from the code.

In [ ]:
#@title <small><small> { display-mode: "form" }

# mute warnings/erros in console
RDLogger.logger().setLevel(RDLogger.CRITICAL)

molecule = "NO2"
GEOMINFO = load_n2o4_2no2()[4]
# ------------------------------------------------
if "MOLECULES_ID" not in globals(): MOLECULES_ID  = {k:None for k in MOLECULES}
# ------------------------------------------------
MOLECULES_ID[molecule] = input(rf"insert the SMILES code for {molecule:s} : ").strip()
if MOLECULES_ID[molecule].startswith('"'): MOLECULES_ID[molecule] = MOLECULES_ID[molecule][1:]
if MOLECULES_ID[molecule].endswith('"')  : MOLECULES_ID[molecule] = MOLECULES_ID[molecule][:-1]
print("")

# Get geometry
try:
   mol_id    = MOLECULES_ID[molecule]
   xyz_guess = files_of_interest(molecule)[0]
   print(rf"Retrieving geometry for: {str(mol_id):s}")
   print("")
   symbols,coords,smiles = rdkit_smiles2geom(mol_id)
   data_2_xyz(symbols,coords,xyz_guess,smiles)
   print(rf"     - geometry stored in '{xyz_guess:s}'")
   info = geometric_info_xyz(xyz_guess,GEOMINFO[molecule])
   print(info)
   print("")
   view = create_visualization_xyz(xyz_guess)
   view.show()

except:
   print(rf"ERROR! Something went wrong! Did you introduce the correct SMILES??")
   print("")

#####
The molecular structures obtained so far are **not** optimized; they do not correspond to the **minimum-energy geometries**.

Before performing any optimization, however, we must specify for each molecule both its total charge and the number of unpaired electrons. In our case:

* N$_2$O$_4$: **neutral** molecule with **no** unpaired electrons
* NO$_2$ &nbsp;: **neutral** molecule with **one** unpaired electron

Run the next cell and enter the value for these variables:

In [ ]:
#@title <small><small> { display-mode: "form" }

CHARGES,UNPAIREDS = {},{}
print("Insert the total charge (a.u.) and number of unpaired electrons for each molecule (integers only):")
print("")
for molecule in MOLECULES:
    print(rf"   * {molecule:s}")
    charge   = input("     total molecular charge [in au] = ").strip()
    unpaired = input("     number of unpaired electrons   = ").strip()
    CHARGES[molecule]   = int(charge)
    UNPAIREDS[molecule] = int(unpaired)
    print("")

#### **(b) DFT calculations**

#####

Now, we are ready to perform the quantum-chemistry calculations using **PySCF**[[🌐]](https://pyscf.org). After optimizing the geometries, we will also compute the **Hessian matrix** to verify that each structure corresponds to a true minimum (all vibrational frequencies are real), rather than a different type of stationary point. Moreover, these vibrational frequencies are required to obtain the vibrational partition function, which is essential for evaluating thermodynamic quantities.

We will test the following DFT levels of theory:

* B3LYP-D4
* O3LYP-D4
* PBE0-D4

with the **6-31G*** basis set. Run the next cell to load these methods and the basis set.

<br>

**Note:** In PySCF, the selected basis set corresponds to "**6-31G*** **5d**" in _Gaussian_ electronic structure software [[🌐]](https://gaussian.com/basissets/).

In [ ]:
#@title <small><small> { display-mode: "form" }

# =====================================================
# Classroom setting DFTGRID = 2 (short runtime).
# For higher accuracy, set DFTGRID = 5 (slower).
#------------------------------------------------------
DFTGRID       = 2
# =====================================================

# =====================================================
FUNCTIONALS   = "b3lyp-d4,o3lyp-d4,pbe0-d4".split(",")
BASIS         = "6-31g*"
#------------------------------------------------------
print(f"The following DFT functionals will be tested in combination with the {BASIS.upper():s} basis set:")
for functional in FUNCTIONALS:
    print(f"   * {functional.upper():s}")
print("")
print(f"Grid level is set to: {DFTGRID:d}")
print("")
#------------------------------------------------------
if "DFTDATA" not in globals(): DFTDATA = {k:{}   for k in MOLECULES}
# =====================================================

Now run the next three cells in sequence to perform the quantum-chemistry calculations.

 * ⏳ Each cell typically takes **~6 minutes** to run. ⏳

<br>

**Note:** To keep runtimes manageable, we use a coarse DFT integration grid. More accurate results are obtained with a **level 5** grid. You can switch to it by changing `DFTGRID` from **2** to **5** in the previous cell and re-running it; this usually increases the runtime to **~10–15 min per cell**.



In [ ]:
#@title <small>💻 Optimizing guess structures with B3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[0],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],DFTGRID,bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],DFTGRID,[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

In [ ]:
#@title <small>💻 Optimizing guess structures with O3LYP-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[1],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],DFTGRID,bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],DFTGRID,[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

In [ ]:
#@title <small>💻 Optimizing guess structures with PBE0-D4 <small> { display-mode: "form" }

key = (FUNCTIONALS[2],BASIS)
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print(rf" - Grid level: {DFTGRID:d}")
print("")

t1 = time.time()
for molecule in MOLECULES:
    print(rf" * Molecule: {molecule:s}")
    DFTDATA[molecule][key] = optimize_and_freqs(molecule,UNPAIREDS[molecule],CHARGES[molecule],key[0],key[1],DFTGRID,bsym=True)
    print("")
    pyscf_printdata(DFTDATA[molecule][key])
    pyscf_download(molecule,key[0],key[1],DFTGRID,[1])
    print("")
t2 = time.time()
print(" - total elapsed time: %.2f minutes"%((t2-t1)/60))

#####
Run the cell below to **visualize** the optimized geometries. Again, the views are interactive; you can rotate and inspect the molecule freely.

In [ ]:
#@title <small><small> { display-mode: "form" }

GEOMINFO = load_n2o4_2no2()[4]
print("")
for molecule in MOLECULES:

    html_blocks = []
    print(rf"==> molecule: {molecule:s}")
    for functional in FUNCTIONALS:

        xyz_opt = files_of_interest(molecule,functional,BASIS,DFTGRID)[1]
        if not os.path.exists(xyz_opt): continue
        # generate visualization of molecule
        view = create_visualization_xyz(xyz_opt)
        # read xyz file and get some interesting geometric features
        info  = functional+"/"+BASIS + "\n"
        info += geometric_info_xyz(xyz_opt,GEOMINFO[molecule])
        # save for later visualization
        html_blocks.append(f"""
            <div style='text-align:center; margin:10px;'>
                {view._make_html()}
                <div style='font-size:16px; font-weight:500; margin-top:-2px;white-space:pre-line;'>
                    {info}
                </div>
            </div>
        """)

    # Displays all viewers side by side
    html_code = "<div style='display:flex; gap:100px; width:100%;'>" + "".join(html_blocks) + "</div>"
    display(HTML(html_code))
    print("")

#####

PySCF does not always return the correct symmetry number ($\sigma$) due to numerical inaccuracies [[🌐]](https://link.springer.com/article/10.1007/s00214-007-0328-0). Therefore, before moving to (c) and compute the partition functions, we must ensure that the symmetry numbers are correct. For the species involved in the reaction studied here, the appropriate values are:

* N$_2$O$_4$ ==> D2h symmetry ==> $\sigma = 4$
* NO$_2$ &nbsp;  ==> C2v symmetry ==> $\sigma = 2$

Verify these values in the preceding optimization cells, and run the following cell if any symmetry number needs to be corrected.

In [ ]:
#@title <small>💻 Correct symmetry numbers <small> { display-mode: "form" }
print("Insert the rotational symmetry number for each molecule:")
print("")
for molecule in MOLECULES:
    sigma = int(input(rf"   * sigma({molecule:s}) = ").strip())
    print("")
    for functional in FUNCTIONALS:
        key = (functional,BASIS)
        try:
          sigma_old = DFTDATA[molecule][key]["rotsigma"]
          print(rf"     {functional.upper():8s}: {sigma_old:d} --> {sigma:d}")
          DFTDATA[molecule][key]["rotsigma"] = sigma
        except:
          print(rf"     data not found for {functional.upper():s}...")
    print("")

#### **(c) Calculation of partition functions and thermodynamic functions**


#####
We can now evaluate the partition functions of all species participating in the reaction at 298.15 K. From them, the thermodynamic state functions can be determined.

In [ ]:
#@title <small><small> { display-mode: "form" }

TABLE  = " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "  FUNCTIONAL/BASIS_SET | DELTA_r{U}^o | DELTA_r{H}^o | DELTA_r{S}^o | DELTA_r{G}^o |   K_p^o   " + "\n"
TABLE += " ----------------------|--------------|--------------|--------------|--------------|-----------" + "\n"

print("Partition functions and zero-point–corrected total energies (E0 + ZPE); energies in hartree.")
print("")

print(f"    * Reference temperature: {TREF:.2f} K\n")
DFT_REFTHERMO = {}
for functional in FUNCTIONALS:
    key   = (functional,BASIS)
    level = rf"{key[0].upper():8s}/{key[1].upper():s}"
    allok = True

    SUBTABLE  = "    --------------------------------------------------------------------------------------------"+"\n"
    SUBTABLE += "      molecule |   pfn tr    |  pfn rot   |  pfn vib   |  pfn ele   |  pfn tot   |   E0 + ZPE   "+"\n"
    SUBTABLE += "    -----------|-------------|------------|------------|------------|---------------------------"+"\n"

    DUo,DHo,DSo,DGo  = 0.,0.,0.,0.
    for nu_i,molecule in zip(NUS,MOLECULES):

        output_frq = files_of_interest(molecule,key[0],key[1],DFTGRID)[3]
        if not os.path.exists(output_frq):
           allok = False
           break
        # calculate thermodynamics
        try: Uo, Ho, So, Go, line = compute_thermodynamics(TREF,molecule,key,DFTDATA)
        except: allok = False; break
        SUBTABLE += rf"      {molecule:8s} | " + line + "\n"

        # reaction magnitudes
        DUo += nu_i * Uo
        DHo += nu_i * Ho
        DSo += nu_i * So
        DGo += nu_i * Go
    SUBTABLE += "    --------------------------------------------------------------------------------------------"+"\n"
    if not allok: continue
    print(rf"    ==> {level:s} <==")
    print("")
    print(SUBTABLE)
    # Equilibrium constant
    Kp = np.exp(-DGo/k_B/TREF)
    # Print info
    if Kp > 1E4: ff = "9.2E"
    else       : ff = "9.3f"
    TABLE += rf"  {level:12s}      | {DUo*NA/1000:12.2f} | {DHo*NA/1000:12.2f} | {DSo*NA:12.2f} | {DGo*NA/1000:12.2f} | {Kp:{ff:s}} " + "\n"
    # save reaction magnitudes
    DFT_REFTHERMO[key] = TREF,DUo*NA,DHo*NA,DSo*NA,DGo*NA
TABLE += " ----------------------------------------------------------------------------------------------" + "\n"
TABLE += "\n"
TABLE += "     units of DELTA_r{U}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{H}^o ==> kJ/mol"   + "\n"
TABLE += "     units of DELTA_r{S}^o ==>  J/mol/K" + "\n"
TABLE += "     units of DELTA_r{G}^o ==> kJ/mol"   + "\n"

print("")
print(TABLE)
print("")

#####

The experimental value for $\Delta_{r}G^\circ$(298.15 K) is 4.74 kJ/mol, and O3LYP-D4/6-31G* provides the closest agreement among the tested levels of theory. Therefore, we will use this functional for the final step (a.3).

#### **(d) Temperature dependence of $\Delta_{r}G^\circ$: $\Delta_{r}C_{p}^\circ$ and the vibrational degrees of freedom**

#####

The calculation of $\Delta_{r}G^\circ$ in (c) can be performed at temperatures other than 298.15 K. Let us check how the O3LYP-D4/6-31G* level predicts $\Delta_{r}G^\circ$ over the 200-400 K temperature range.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key  = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
T    = np.linspace(TMIN,TMAX,num=NPOINTST)
#------------------------------------------------------------------------------#
# Calculate D_{r}G^o at different temperatures using Statistical Thermodynamics
print(rf"Computing Delta_{{r}}G^o from {T[0]:.2f} to {T[-1]:.2f} K using Statistical Mechanics...")
print("")
print(rf" - Functional: {key[0].upper():s}")
print(rf" - Basis set : {key[1].upper():s}")
print("")
DGo_T = []
for Ti in list(T):
    DGo_i = [nu_i * compute_thermodynamics(Ti,molecule,key,DFTDATA)[3] for nu_i,molecule in zip(NUS,MOLECULES)]
    DGo_i = float(sum(DGo_i)*NA)
    DGo_T.append(DGo_i)
DGo_T = np.array(DGo_T)

# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]

# Plot data
plot_DGo_T_statmech(T,DGo_T,None,None,TREF,DGo_qc,key)

#####

As you may recall from the first part of this Notebook, this temperature dependence can be written as:

$$
\Delta_{r} G^{\circ}(T) = \Delta_{r} H^{\circ}(T_{\rm ref})-T \cdot \Delta_{r} S^{\circ}(T_{\rm ref})+\Delta_{r} C_p^\circ \cdot \left[ T-T_{\rm ref}+T \cdot \ln \left(\frac{T_{\rm ref}}{T}\right)\right]
\tag{1}
$$

under the assumption that $\Delta_{r}C_{p}^\circ$ is constant. Note that we already calculated all quantities at the reference temperature ($T_{\rm ref}$ = 298.15 K), except for $\Delta_{r}C_{p}^\circ$ itself.

Next, we will explore how the choice of $\Delta_{r}C_{p}^\circ$ influences the predicted temperature dependence of $\Delta_{r} G^{\circ}(T)$, and which value best reproduces the numerical results obtained in the previous cell. Run the following cell as many times as you want, enter a trial value of $\Delta_{r}C_{p}^\circ$, and compare the curve from Eq. (1) with the previously computed one.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
T   = np.linspace(TMIN,TMAX,num=NPOINTST)
# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]
#------------------------------------------------------------------------------#

# Values at optimized results (and other values)
print(" We can see how data fit according to the value of Delta_{r}C_{p}^o = constant * R")
DCP = float(input("   - insert value for the constant: ").strip())

DGo_model = get_DGo(T,(DHo_qc,DSo_qc,DCP*R,TREF))
RMS       = calculate_rms(DGo_T,DGo_model)/1000
print("   * Delta_{r}C_{p}^o = %5.2f*R ==> RMS = %.3f kJ/mol"%(DCP,RMS))
print("")
plot_DGo_T_statmech(T,DGo_T,DGo_model,DCP*R,TREF,DGo_qc,key)
print()

#####

The optimal value of $\Delta_{r} C_p^\circ$ can be obtained directly by fitting the computed $\Delta_{r} G^\circ(T)$ values (calculated using Statistical Thermodynamics) to Eq. (1). In other words, instead of choosing  $\Delta_{r} C_p^\circ$ manually, one could determine the value that minimizes the deviation between the analytical expression, Eq. (1), and the data, yielding the best agreement across the temperature range.

To do this, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#
T   = np.linspace(TMIN,TMAX,num=NPOINTST)
# Data at reference temperature
TREF,DUo_qc,DHo_qc,DSo_qc,DGo_qc = DFT_REFTHERMO[key]
#------------------------------------------------------------------------------#

# minimize
print("Obtaining best value for Delta_{r}C_{p}^o by least squares...")
print("")
res       = minimize_scalar(sum_squared_errors,bounds=(-5,5),method='bounded',args=(T,DGo_T,TREF,DHo_qc,DSo_qc))
DCP_best  = res.x

# Values at optimized results (and other values)
DGo_model = get_DGo(T,(DHo_qc,DSo_qc,DCP_best*R,TREF))
RMS       = calculate_rms(DGo_T,DGo_model)/1000

print("   * best fit --> Delta_{r}C_{p}^o = %5.2f*R   [RMS = %.3f kJ/mol]"%(DCP_best,RMS))
print("")

# Plotting
plot_DGo_T_statmech(T,DGo_T,DGo_model,DCP_best*R,TREF,DGo_qc,key)
print()

#####
The optimal value of $\Delta_{r} C_p^\circ$ can also be estimated from statistical mechanics. In this approach, the molar heat capacity at constant pressure of a given molecule is written as:

$$
C_{p,m} = R + \frac{3 + n^{R^\ast} + 2 \cdot n^{V^\ast}}{2} \cdot R
\tag{8}
$$

where $n^{R^\ast}$ and $n^{V^\ast}$ are the effective rotational and vibrational degrees of freedom (dofs), respectively. The term 3 accounts for the traslational dofs. For linear molecules, $n^{R^\ast}=2$; otherwise, $n^{R^\ast}=3$.

The vibrational contribution, $n^{V^\ast}$, is obtained by summing the contributions from each normal mode (frequency $\nu_j$):

$$
n^{V^\ast} = \sum_j n^{V^\ast}_j
\tag{9}
$$

where each mode contributes:

$$
n^{V^\ast}_j = \left( \frac{\theta_j^V}{T} \cdot \frac{\exp(-\theta_j^V/2T)}{1-\exp(-\theta_j^V/T)} \right)^2
\tag{10}
$$

In the previous equation, $\theta_j^V = h \nu_j / k_B$, where $h$ and $k_B$ are the Planck and the Boltzmann constants, respectively.

To explore how $n^{V^\ast}_j$ depends on the vibrational frequency, $\nu_j$, run the following cell.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#

# Find limits for the frequencies
freqmin = +float("inf")
freqmax = -float("inf")
for molecule in DFTDATA.keys():
    freqs = DFTDATA[molecule][key]["freqs"]
    freqmin = min(freqmin,min(freqs))
    freqmax = max(freqmax,max(freqs))
freqmin_cm = np.floor(freqmin/100)
freqmax_cm = np.floor(freqmax/100)

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args        = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
freq_slider = w.FloatSlider(value=freqmin_cm,min=freqmin_cm,max=freqmax_cm,step=0.01,description=r'freq [1/cm]', **args)
ui       = w.VBox([freq_slider])

# -------- download button --------
btn      = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,"dof"))

# -------- slider ---> function --------
# notice that P is converted from bar to Pa with *1E5
out = w.interactive_output(lambda freq: plot_vib_average(TMIN,TMAX,freqmin_cm,freqmax_cm,freq), {'freq': freq_slider})
display(w.VBox([ui, btn]), out)

#####

Run the following cell to compute $C_{p,m}$ for each molecule involved in the reaction, as well as $\Delta_{r} C_p^\circ$. You will see that the value obtained is the same as the optimal value.

In [ ]:
#@title <small><small> { display-mode: "form" }

#------------------------------------------------------------------------------#
key = (FUNCTIONALS[1],BASIS)
#------------------------------------------------------------------------------#

print("-------------------------------------------")
print(" Molecule | rot dof  | vib dof  |  C_{p,m} ")
print("-------------------------------------------")
DCP = 0.0
for i,molecule in enumerate(MOLECULES):
    # check if data is available
    if molecule not in DFTDATA: continue
    if key      not in DFTDATA[molecule]: continue
    # rotational dofs
    if DFTDATA[molecule][key]["islinear"]: rdof = 2
    else                                 : rdof = 3
    # calculate vdof and Cp
    freqs_cm = [ii/100 for ii in DFTDATA[molecule][key]["freqs"]]
    vdof     = sum([vib_contri_avera(freq_cm,TMIN,TMAX) for freq_cm in freqs_cm])
    Cp       = R + (3+rdof+2*vdof)/2*R
    DCP     += Cp*NUS[i]
    print(rf" {molecule:7s}  |  {rdof:6d}  |  {vdof:6.3f}  | {Cp/R:6.3f}*R  ")
print("-------------------------------------------")
print(rf"             ==> Delta_r{{C_p}}^o = {DCP/R:6.2f}*R")
print("")

### **3. A Chemical Kinetics Perspective**  


####
In this section, we will analyze the connection between **chemical equilibrium** and **chemical kinetics**.

Our goal is to explore how equilibrium is reached as a _dynamic process_. We will consider the two classical scenarios: constant-_T,p_ and constant-_T,V_.

####
**(a) Kinetic background**


#####
To connect kinetics with thermodynamics, we require the forward ($k_{fw}$) and backward ($k_{bw}$) rate constants. For the reaction here studied, these rate constants are related to the equilibrium constant according to:

$$
K_p^\circ(T) \cdot \frac{p^\circ}{RT} = \frac{k_{fw}}{k_{bw}}
 \tag{11}
$$

where $p^\circ = 1$ bar is the standard pressure. Thus, if the temperature dependence of $k_{fw}(T)$ is known, the backward rate constant follows immediately.

For the forward reaction, we can adopt an Arrhenius-type expression:

$$
k_{fw} = A \cdot e^{-B/T}
 \tag{12}
$$

where $A$ is the pre-exponential factor and $B$ is related to the activation energy. In particular, we will use the parametrization reported in _Atmos. Chem. Phys._ **4**, 1461-1738 (2004)[[🌐]](https://doi.org/10.5194/acp-4-1461-2004):

$$
k_{fw} = 1.15 \cdot 10^{16}  \cdot e^{-6460/T} \;\;\; s^{-1}
 \tag{13}
$$

Although this expression is formally valid only in the 250–300 K range and corresponds to the high-pressure limit, we will apply it over the full temperature interval for pedagogical purposes.

<br>

Execute the following cell to load the values of $A$ and $B$.
[_You may adjust any parameter in the cell prior to execution if you wish to explore alternative data or hypothetical scenarios._]

In [ ]:
#@title <small> <small> { display-mode: "form" }
arrhenius_A = 1.15E16 # in 1/s
arrhenius_B = 6460    # in K
print(rf"Value for A: {arrhenius_A} 1/s")
print(rf"Value for B: {arrhenius_B} K")

#### **(b) Carrying out the experiments**

#####
Use the following cell to configure and analyze a scenario by selecting the initial temperature, pressure, and volume of the system. You may also specify the initial mole fraction of the reactant (N$_2$O$_4$) and choose whether the experiment is performed under constant-_T,p_ or constant-_T,V_ conditions.
The plots will update dynamically as you adjust the initial conditions using the sliders.

In [ ]:
#@title <small> <small> { display-mode: "form" }

# Enable Colab’s custom widget manager, allowing interactive ipywidgets to function correctly
output.enable_custom_widget_manager()

# -------- Sliders --------
args      = dict(layout=w.Layout(width='600px'),style={'description_width': '150px'},continuous_update=True,readout_format='.2f')
T_slider  = w.FloatSlider(value=TREF ,min=TMIN,max=  TMAX,step=1.00,description=r'T [K]'  , **args)
P_slider  = w.FloatSlider(value= 1.00,min=0.01,max=  3.00,step=0.01,description=r'p [bar]', **args)
V_slider  = w.FloatSlider(value=24.78,min=2.00,max=250.00,step=0.01,description=r'V [L]'  , **args)
yA_slider = w.FloatSlider(value= 1.00,min=0.00,max=  1.00,step=0.01,description=r'y(N2O4)', **args)

# -------- Scenario selector --------
selector = w.RadioButtons(
    options=[('constant-T,V', 'kinVT'),
             ('constant-T,p', 'kinPT')],
    style={'description_width': '100px'},
    layout=w.Layout(width='260px', margin='0 0 0 40px')
)

# -------- download button --------
btn = w.Button(description='Download current figure', icon='download', button_style='primary',layout=w.Layout(width='200px', height='30px'))
btn.on_click(lambda b: on_download_clicked(b,selector.value))

# -------- slider ---> function --------
# P*1E+5 : bar --> Pa
# V*1E-3 : L   --> m3
string = w.Output()
out    = w.interactive_output(lambda T0, P0, V0, yA0, scenario: kinetics_N2O4(T0, P0*1E5, V0*1E-3,yA0,scenario), {'T0': T_slider, 'P0': P_slider,'V0': V_slider,'yA0': yA_slider,'scenario':selector})

# Print sliders and selector
print("Set the initial conditions:")
ui = w.VBox([T_slider,P_slider,V_slider,yA_slider])
display(ui)
print("Select scenario:")
display(w.VBox([selector]))

# Print download button and plot
display(w.VBox([btn,out]))

#####

By running the next cell, the information about both the initial state and the equilibrium state corresponding to the most recent experiment configured in the previous cell will be displayed.

In [ ]:
#@title <small> <small> { display-mode: "form" }
print(last_info)